In [1]:
import os
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [2]:
def find_project_root(start=None):
    """
    .env 파일을 기준으로 프로젝트의 최상위 경로를 탐색합니다.

    args:
        - start: 프로젝트 루트 탐색을 시작할 경로
                 None인 경우 현재 작업 경로에서 탐색을 시작합니다.

    return:
        - .env 파일이 존재하는 프로젝트 루트의 Path 객체
    """
    current = Path(start or Path.cwd()).resolve()

    while current != current.parent:
        if (current / ".env").exists():
            return current

        current = current.parent

    raise FileNotFoundError(
        "프로젝트 루트의 .env 파일을 찾을 수 없습니다."
    )


def create_mysql_engine(host, port, user, password, database):
    """
    지정한 MySQL 데이터베이스에 연결할 SQLAlchemy Engine을 생성합니다.

    args:
        - host: MySQL 서버 주소
        - port: MySQL 서버 포트
        - user: MySQL 사용자명
        - password: MySQL 비밀번호
        - database: 연결할 데이터베이스명

    return:
        - 생성된 SQLAlchemy Engine 객체
    """
    db_url = URL.create(
        drivername="mysql+pymysql",
        username=user,
        password=password,
        host=host,
        port=port,
        database=database,
        query={"charset": "utf8mb4"}
    )

    return create_engine(
        db_url,
        pool_pre_ping=True
    )

In [3]:
# 프로젝트 루트 탐색
PROJECT_ROOT = find_project_root()

# 환경변수 로드
load_dotenv(PROJECT_ROOT / ".env")

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /Users/lee-jongyoon/Documents/bigcontest


In [4]:
# MySQL 접속정보 설정
DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT"))
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

In [5]:
# STG 데이터베이스 연결
stg_engine = create_mysql_engine(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database="bigcontest_stg"
)

In [6]:
# 현재 연결된 데이터베이스 확인
with stg_engine.connect() as conn:
    database = conn.execute(
        text("SELECT DATABASE();")
    ).scalar()

print("Connected Database:", database)

Connected Database: bigcontest_stg


In [7]:
# 카드 결제 데이터 로드
card = pd.read_sql(
    """
    SELECT *
    FROM card_topic1;
    """,
    stg_engine
)

print("card shape:", card.shape)

card.head()

card shape: (1044710, 13)


,raw_id,TA_YMD,TIME_GB,MCT_SGG_CD,MCT_RY_CD,SEX_CCD,AGE_CCD,TS_AT,USE_CNT,source_file,source_row_num,raw_loaded_at,stg_loaded_at
0,1,2025-07-04,12_17,서울 강남구,가구,법인,법인,15477163,16,신한카드_빅콘테스트2026_데이터1.txt,1,2026-09-22 09:37:27,2026-09-22 09:40:35
1,2,2025-07-21,12_17,서울 강남구,가구,법인,법인,308451,38,신한카드_빅콘테스트2026_데이터1.txt,2,2026-09-22 09:37:27,2026-09-22 09:40:35
2,3,2025-07-03,12_17,서울 강남구,가구,여성,20 대,78970,5,신한카드_빅콘테스트2026_데이터1.txt,3,2026-09-22 09:37:27,2026-09-22 09:40:35
3,4,2025-07-03,18_23,서울 강남구,가구,남성,20 대,271220,5,신한카드_빅콘테스트2026_데이터1.txt,4,2026-09-22 09:37:27,2026-09-22 09:40:35
4,5,2025-07-06,18_23,서울 강남구,가구,여성,20 대,260872,5,신한카드_빅콘테스트2026_데이터1.txt,5,2026-09-22 09:37:27,2026-09-22 09:40:35


In [8]:
# 성·연령별 유동인구 데이터 로드
flow_age = pd.read_sql(
    """
    SELECT *
    FROM flow_age;
    """,
    stg_engine
)

print("flow_age shape:", flow_age.shape)

flow_age.head()

flow_age shape: (596252, 21)


,raw_id,STD_YM,BLOCK_CD,X_COORD,Y_COORD,MAN_FLOW_POP_CNT_10G,MAN_FLOW_POP_CNT_20G,MAN_FLOW_POP_CNT_30G,MAN_FLOW_POP_CNT_40G,MAN_FLOW_POP_CNT_50G,...,WMAN_FLOW_POP_CNT_10G,WMAN_FLOW_POP_CNT_20G,WMAN_FLOW_POP_CNT_30G,WMAN_FLOW_POP_CNT_40G,WMAN_FLOW_POP_CNT_50G,WMAN_FLOW_POP_CNT_60GU,source_file,source_row_num,raw_loaded_at,stg_loaded_at
0,1,202507,11230510101020000001,956633.991787,1.947479e+06,0.00,0.05,0.05,0.08,0.10,...,0.03,0.08,0.05,0.05,0.05,0.05,flow_age_pop_202507.csv,1,2026-09-22 09:18:06,2026-09-22 09:40:20
1,2,202507,11230510101020000001,956683.991787,1.947429e+06,0.00,0.08,0.10,0.15,0.15,...,0.00,0.08,0.08,0.08,0.05,0.08,flow_age_pop_202507.csv,2,2026-09-22 09:18:06,2026-09-22 09:40:20
2,3,202507,11230510101020000001,956683.991787,1.947479e+06,0.00,0.05,0.05,0.10,0.10,...,0.03,0.05,0.05,0.05,0.05,0.08,flow_age_pop_202507.csv,3,2026-09-22 09:18:06,2026-09-22 09:40:20
3,4,202507,11230510101020000001,956683.991787,1.947579e+06,0.03,0.08,0.15,0.15,0.18,...,0.00,0.08,0.10,0.10,0.10,0.08,flow_age_pop_202507.csv,4,2026-09-22 09:18:06,2026-09-22 09:40:20
4,5,202507,11230510101020000001,956733.991787,1.947379e+06,0.48,0.69,0.97,1.19,1.22,...,0.46,0.71,0.81,0.89,0.79,0.86,flow_age_pop_202507.csv,5,2026-09-22 09:18:06,2026-09-22 09:40:20


In [9]:
# 시간대별 유동인구 데이터 로드
flow_time = pd.read_sql(
    """
    SELECT *
    FROM flow_time;
    """,
    stg_engine
)

print("flow_time shape:", flow_time.shape)

flow_time.head()

flow_time shape: (663350, 33)


,raw_id,STD_YM,BLOCK_CD,X_COORD,Y_COORD,TMST_00,TMST_01,TMST_02,TMST_03,TMST_04,...,TMST_18,TMST_19,TMST_20,TMST_21,TMST_22,TMST_23,source_file,source_row_num,raw_loaded_at,stg_loaded_at
0,1,202507,32010350100010000001,1.000584e+06,1.969379e+06,0.03,0.03,0.03,0.00,0.00,...,0.10,0.08,0.05,0.05,0.03,0.03,flow_time_pop_202507.csv,1,2026-09-22 09:19:38,2026-09-22 09:40:24
1,2,202507,32010350100010000001,1.000584e+06,1.969429e+06,0.03,0.03,0.03,0.00,0.03,...,0.08,0.08,0.05,0.05,0.03,0.03,flow_time_pop_202507.csv,2,2026-09-22 09:19:38,2026-09-22 09:40:24
2,3,202507,32010350100010000001,1.000584e+06,1.969529e+06,0.03,0.03,0.03,0.03,0.03,...,0.08,0.08,0.05,0.05,0.03,0.03,flow_time_pop_202507.csv,3,2026-09-22 09:19:38,2026-09-22 09:40:24
3,4,202507,32010350100010000001,1.000584e+06,1.969579e+06,0.00,0.00,0.00,0.00,0.00,...,0.03,0.03,0.00,0.00,0.00,0.00,flow_time_pop_202507.csv,4,2026-09-22 09:19:38,2026-09-22 09:40:24
4,5,202507,32010350100010000001,1.000584e+06,1.969629e+06,0.03,0.03,0.00,0.00,0.00,...,0.08,0.08,0.05,0.03,0.03,0.03,flow_time_pop_202507.csv,5,2026-09-22 09:19:38,2026-09-22 09:40:24


In [10]:
# 요일별 유동인구 데이터 로드
flow_wkdy = pd.read_sql(
    """
    SELECT *
    FROM flow_wkdy;
    """,
    stg_engine
)

print("flow_wkdy shape:", flow_wkdy.shape)

flow_wkdy.head()

flow_wkdy shape: (842073, 16)


,raw_id,STD_YM,BLOCK_CD,X_COORD,Y_COORD,FLOW_POP_CNT_MON,FLOW_POP_CNT_TUS,FLOW_POP_CNT_WED,FLOW_POP_CNT_THU,FLOW_POP_CNT_FRI,FLOW_POP_CNT_SAT,FLOW_POP_CNT_SUN,source_file,source_row_num,raw_loaded_at,stg_loaded_at
0,1,202507,32010350100010000001,1.000584e+06,1.969379e+06,1.60,1.55,2.03,1.98,2.39,0.0,0.0,flow_wkdy_pop_202507.csv,1,2026-09-22 09:22:19,2026-09-22 09:40:30
1,2,202507,32010350100010000001,1.000584e+06,1.969429e+06,2.03,2.03,1.98,2.06,1.98,0.0,0.0,flow_wkdy_pop_202507.csv,2,2026-09-22 09:22:19,2026-09-22 09:40:30
2,3,202507,32010350100010000001,1.000584e+06,1.969529e+06,1.78,1.70,2.06,2.01,2.31,0.0,0.0,flow_wkdy_pop_202507.csv,3,2026-09-22 09:22:19,2026-09-22 09:40:30
3,4,202507,32010350100010000001,1.000584e+06,1.969579e+06,0.33,0.38,0.30,0.30,0.30,0.0,0.0,flow_wkdy_pop_202507.csv,4,2026-09-22 09:22:19,2026-09-22 09:40:30
4,5,202507,32010350100010000001,1.000584e+06,1.969629e+06,1.91,1.91,1.85,1.91,1.85,0.0,0.0,flow_wkdy_pop_202507.csv,5,2026-09-22 09:22:19,2026-09-22 09:40:30
